In [5]:
import pandas as pd

In [6]:
from pathlib import Path
p = Path.cwd().parent
print(p)

c:\Users\Asus\Desktop\churn_prediction


In [7]:
train_df = pd.read_csv(
    f"{p}/dataset/processed/train_processed.csv"
)

test_df = pd.read_csv(
    f"{p}/dataset/processed/test_processed.csv"
)


In [10]:
X_train = train_df.drop(
    "Churn Value",
    axis=1
)

y_train = train_df["Churn Value"]

In [11]:
X_test = test_df.drop(
    "Churn Value",
    axis=1
)

y_test = test_df["Churn Value"]

In [12]:
from sklearn.linear_model import LogisticRegression

In [13]:
from sklearn.ensemble import RandomForestClassifier

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

In [15]:
models = {

    "Logistic Regression":
        LogisticRegression(),

    "Random Forest":
        RandomForestClassifier(),

    "Gradient Boosting":
        GradientBoostingClassifier()
}

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(X_train, y_train, X_test, y_test, models):

    report = {}

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        precision = precision_score(y_test, y_pred)

        recall = recall_score(y_test, y_pred)

        f1 = f1_score(y_test, y_pred)

        report[model_name] = {
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        }

    return report

In [18]:
import sys
from pathlib import Path
p = Path.cwd().parent
sys.path.append(str(p))
from src.utils import evaluate_model   

In [19]:
model_report = evaluate_model(
    X_train,
    y_train,
    X_test,
    y_test,
    models
)

print(model_report)

{'Logistic Regression': {'Accuracy': 0.8019872249822569, 'Precision': 0.6426426426426426, 'Recall': 0.5721925133689839, 'F1 Score': 0.6053748231966054}, 'Random Forest': {'Accuracy': 0.7913413768630234, 'Precision': 0.6379310344827587, 'Recall': 0.4946524064171123, 'F1 Score': 0.5572289156626506}, 'Gradient Boosting': {'Accuracy': 0.7998580553584103, 'Precision': 0.6493506493506493, 'Recall': 0.5347593582887701, 'F1 Score': 0.5865102639296188}}


In [20]:
best_model_score = 0

best_model_name = None

best_model = None

for model_name, metrics in model_report.items():

    if metrics["F1 Score"] > best_model_score:

        best_model_score = metrics["F1 Score"]

        best_model_name = model_name

        best_model = models[model_name]

In [21]:
print(f"Best Model: {best_model_name}")

print(f"Best F1 Score: {best_model_score}")

Best Model: Logistic Regression
Best F1 Score: 0.6053748231966054


In [22]:
import os
class ModelTrainerConfig:

    trained_model_path = os.path.join(
        "artifacts",
        "model.pkl"
    )

In [23]:
from sklearn.metrics import classification_report

y_pred = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.64      0.57      0.61       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409



In [24]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[916 119]
 [160 214]]


In [26]:
print(ModelTrainerConfig.trained_model_path)

artifacts\model.pkl


In [27]:
y_prob = best_model.predict_proba(X_test)[:,1]

roc_auc = roc_auc_score(
    y_test,
    y_prob
)
print("ROC-AUC:", roc_auc)

ROC-AUC: 0.8484357642925417


In [28]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[916 119]
 [160 214]]


In [29]:
from sklearn.model_selection import RandomizedSearchCV

In [30]:
param_grid = {

    "n_estimators": [100,200,300,500],

    "max_depth": [5,10,15,20,None],

    "min_samples_split": [2,5,10],

    "min_samples_leaf": [1,2,4],

    "class_weight": [
        None,
        "balanced"
    ]
}

In [31]:
rf = RandomForestClassifier(
    random_state=42
)

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'class_weight': [None, 'balanced'],
                                        'max_depth': [5, 10, 15, 20, None],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='f1', verbose=2)

In [32]:
print(search.best_params_)

{'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 10, 'class_weight': 'balanced'}


In [33]:
best_model = search.best_estimator_

In [34]:
y_pred = best_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.74      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409



In [37]:
import os

print(os.getcwd())
print(ModelTrainerConfig.trained_model_path)
print(os.path.exists("artifacts"))

c:\Users\Asus\Desktop\churn_prediction\src
artifacts\model.pkl
False


In [41]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[2]

trained_model_path = PROJECT_ROOT / "artifacts" / "model.pkl"

In [42]:
print(os.path.abspath(ModelTrainerConfig.trained_model_path))

c:\Users\Asus\Desktop\churn_prediction\artifacts\model.pkl


In [47]:
from src.utils import save_object

save_object(
    ModelTrainerConfig.trained_model_path,
    best_model
)

In [48]:
print(train_df.columns.tolist())

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', 'Churn Value']
